# Prompts

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
print(f'{os.environ['OPENAI_API_KEY'][:20]}...')


sk-proj-DPrT3N-1Ilp9...


In [2]:
"""
prompts class 계층도

BasePromptTemplate --> PipelinePromptTemplate
                       StringPromptTemplate --> PromptTemplate
                                                FewShotPromptTemplate
                                                FewShotPromptWithTemplates
                       BaseChatPromptTemplate --> AutoGPTPrompt
                                                  ChatPromptTemplate --> AgentScratchPadChatPromptTemplate



BaseMessagePromptTemplate --> MessagesPlaceholder
                              BaseStringMessagePromptTemplate --> ChatMessagePromptTemplate
                                                                  HumanMessagePromptTemplate
                                                                  AIMessagePromptTemplate
                                                                  SystemMessagePromptTemplate
"""
None

# import

In [3]:
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# from langchain_openai.chat_models import ChatOpenAI
# from langchain_openai.llms.base import OpenAI
# from langchain_core.prompts.prompt import PromptTemplate
# from langchain_core.prompts.chat import ChatPromptTemplate

from langchain_openai import OpenAI, ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate


In [4]:
chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

# PromptTemplate 포맷팅 방법2가지

In [5]:
# 방법1
t = PromptTemplate.from_template("What is the capital of {country}")
t.format(country="France")

'What is the capital of France'

In [ ]:
# 템플릿을 만드는 이점 또 한가지!
# prompt template 을 디스크에 '저장'하고 'load' 할 수 있기 떼문이다.

# 나중에 LLM 다룰때 prompt 는 매우 중요합니다.
# 대규모 프로젝트에서는 
# - 'prompt 만 만드는 팀'이 있고
# - '코딩하는 팀'이 따로 있을것이다.  
#    데이터베이스나 파일 등에 만들어 저장해 놓은 prompt 를 load 해야 할겁니다

In [6]:
# 방법2
t2 = PromptTemplate(
    template="What is the capital of {country}",
    input_variables=['country'],   # 입력변수들이 무엇인지 알려주어야 한다.
)
t2.format(country='France')

'What is the capital of France'

# 1.FewShotPromptTemplate
모델에 예제(example) 주기

In [ ]:
# 모델에게 '어떻게 대답해야 하는 지에 대한 예제(example)'를 AI 모델에게 주는 것이
# prompt 를 사용해서 '어떻게 대답해야 하는지 알려주는 것'보다 훨씬 좋다

# FewShotPromptTemplate 이 하는 일이 바로 그거다!
# - 이를 통해 예제(샘플)를 형식화(포맷) 할수 있다.
# - 이런 예제들을 데이터베이스등에 저장시켜놓고 활용할수도 있다


In [8]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate

In [9]:
# 모델이 나에게 '이런 식으로 답변해 줬으면 좋겠다' 라고 제시하는 example(예제) 들.

examples = [
  {
    "question": "What do you know about France?",

    # ↓ 원하는 형식의 답변 이다..
    "answer": """
      Here is what I know:
      Capital: Paris
      Language: French
      Food: Wine and Cheese
      Currency: Euro
      """,
  },
  {
    "question": "What do you know about Italy?",
    "answer": """
      I know this:
      Capital: Rome
      Language: Italian
      Food: Pizza and Pasta
      Currency: Euro
      """,
  },
  {
    "question": "What do you know about Greece?",
    "answer": """
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      """,
  },
]

In [9]:
# 예제(example) 없이 전달하면?
chat.invoke("What do you know about France?")

France is a country located in Western Europe. It is known for its rich history, culture, and cuisine. The capital city is Paris, which is famous for landmarks such as the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral.

France is the largest country in the European Union by land area and the third-largest in Europe overall. It has a population of over 67 million people. The official language is French, and the currency is the Euro.

France is a popular tourist destination, attracting millions of visitors each year to its cities, beaches, and countryside. It is also known for its wine production, fashion industry, and art scene.

The country has a long history of influential figures in literature, philosophy, and science, including famous thinkers such as Voltaire, Descartes, and Marie Curie.

France is a founding member of the United Nations, NATO, and the European Union. It has a strong economy and is a major player in global politics and diplomacy.

AIMessage(content='France is a country located in Western Europe. It is known for its rich history, culture, and cuisine. The capital city is Paris, which is famous for landmarks such as the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral.\n\nFrance is the largest country in the European Union by land area and the third-largest in Europe overall. It has a population of over 67 million people. The official language is French, and the currency is the Euro.\n\nFrance is a popular tourist destination, attracting millions of visitors each year to its cities, beaches, and countryside. It is also known for its wine production, fashion industry, and art scene.\n\nThe country has a long history of influential figures in literature, philosophy, and science, including famous thinkers such as Voltaire, Descartes, and Marie Curie.\n\nFrance is a founding member of the United Nations, NATO, and the European Union. It has a strong economy and is a major player in global politics and diplomacy.'

In [10]:
# FewShotPromptTemplate 사용

# 우선 포맷 준비 <- examples 로 포맷팅할거다
example_template = """
    Human: {question}
    AI: {answer}
"""
# ↑ {question} 와 {answer} 는 위 샘플과 동일한 key 를 사용하여 작성.



In [11]:
example_prompt = PromptTemplate.from_template(example_template)
example_prompt

PromptTemplate(input_variables=['answer', 'question'], input_types={}, partial_variables={}, template='\n    Human: {question}\n    AI: {answer}\n')

In [12]:
print(example_prompt.format(**examples[2]))


    Human: What do you know about Greece?
    AI: 
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      



In [13]:
prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,  # 사용할 prompt
    examples=examples,   # 준비된 예시들

    # 여기에 user 의 질문을 넣어줍니다
    suffix="Human: What do you know about {country}",

    # 어떤 입력변수를 suffix 에서 사용할지 지정.
    input_variables=['country']
)

prompt

FewShotPromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, examples=[{'question': 'What do you know about France?', 'answer': '\n      Here is what I know:\n      Capital: Paris\n      Language: French\n      Food: Wine and Cheese\n      Currency: Euro\n      '}, {'question': 'What do you know about Italy?', 'answer': '\n      I know this:\n      Capital: Rome\n      Language: Italian\n      Food: Pizza and Pasta\n      Currency: Euro\n      '}, {'question': 'What do you know about Greece?', 'answer': '\n      I know this:\n      Capital: Athens\n      Language: Greek\n      Food: Souvlaki and Feta Cheese\n      Currency: Euro\n      '}], example_prompt=PromptTemplate(input_variables=['answer', 'question'], input_types={}, partial_variables={}, template='\n    Human: {question}\n    AI: {answer}\n'), suffix='Human: What do you know about {country}')

In [14]:
print(prompt.format(country="Germany"))


    Human: What do you know about France?
    AI: 
      Here is what I know:
      Capital: Paris
      Language: French
      Food: Wine and Cheese
      Currency: Euro
      



    Human: What do you know about Italy?
    AI: 
      I know this:
      Capital: Rome
      Language: Italian
      Food: Pizza and Pasta
      Currency: Euro
      



    Human: What do you know about Greece?
    AI: 
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      


Human: What do you know about Germany


In [ ]:
# step1  example 리스트를 만들고   examples
# step2  FewShotPromptTemplate 에 전달했고 examples=
# step3  어떻게 전달한 예제들을 형식화 할지 알려주었고
# step4  마지막에 질문을 포함시켰다.  suffix, input_variables

# AI 는 우리의 예제들과 똑같은 구조, 형태로 답변하게 될겁니다


In [15]:
chain = prompt | chat
chain

FewShotPromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, examples=[{'question': 'What do you know about France?', 'answer': '\n      Here is what I know:\n      Capital: Paris\n      Language: French\n      Food: Wine and Cheese\n      Currency: Euro\n      '}, {'question': 'What do you know about Italy?', 'answer': '\n      I know this:\n      Capital: Rome\n      Language: Italian\n      Food: Pizza and Pasta\n      Currency: Euro\n      '}, {'question': 'What do you know about Greece?', 'answer': '\n      I know this:\n      Capital: Athens\n      Language: Greek\n      Food: Souvlaki and Feta Cheese\n      Currency: Euro\n      '}], example_prompt=PromptTemplate(input_variables=['answer', 'question'], input_types={}, partial_variables={}, template='\n    Human: {question}\n    AI: {answer}\n'), suffix='Human: What do you know about {country}')
| ChatOpenAI(callbacks=[<langchain_core.callbacks.streaming_stdout.StreamingStdOutCallbackHandler object at 

In [21]:
chain.invoke({'country': 'Germany'})

AI: 
      Here is what I know:
      Capital: Berlin
      Language: German
      Food: Bratwurst and Sauerkraut
      Currency: Euro

AIMessage(content='AI: \n      Here is what I know:\n      Capital: Berlin\n      Language: German\n      Food: Bratwurst and Sauerkraut\n      Currency: Euro', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-3.5-turbo-0125', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--019eedcc-b27e-72d3-ba7a-971b42deb2c0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 151, 'output_tokens': 37, 'total_tokens': 188, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [22]:
chain.invoke({'country': 'Turkey'})

AI: 
      Here is what I know:
      Capital: Ankara
      Language: Turkish
      Food: Kebab and Baklava
      Currency: Turkish Lira

AIMessage(content='AI: \n      Here is what I know:\n      Capital: Ankara\n      Language: Turkish\n      Food: Kebab and Baklava\n      Currency: Turkish Lira', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-3.5-turbo-0125', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--019eedcd-87f7-7d61-b2f8-575ba9396d1f', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 151, 'output_tokens': 37, 'total_tokens': 188, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

# 2.FewShotChatMessagePromptTemplate

In [16]:
from langchain_core.prompts.few_shot import FewShotChatMessagePromptTemplate

In [17]:
examples = [
  {
    "country": "France",
    "answer": """
      Here is what I know:
      Capital: Paris
      Language: French
      Food: Wine and Cheese
      Currency: Euro
      """,
  },
  {
    "country": "Italy",
    "answer": """
      I know this:
      Capital: Rome
      Language: Italian
      Food: Pizza and Pasta
      Currency: Euro
      """,
  },
  {
    "country": "Greece",
    "answer": """
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      """,
  },
]

In [18]:
example_prompt = ChatPromptTemplate.from_messages([
    ('human', "What do you know abount {country}?"),
    ('ai', "{answer}")
])

example_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt, 
    examples=examples,
    # suffix= 등은 필요없다.
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a geography expert, you give short answers."),
    example_prompt, # ✨✨
    ("human", "What do you know abount {country}?"),  # <- 여기에 human message 가 있다.  그래서 suffix 가 필요없던거다. 
])


In [19]:
final_prompt.format_messages(country='Germany')

[SystemMessage(content='You are a geography expert, you give short answers.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What do you know abount France?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='\n      Here is what I know:\n      Capital: Paris\n      Language: French\n      Food: Wine and Cheese\n      Currency: Euro\n      ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What do you know abount Italy?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='\n      I know this:\n      Capital: Rome\n      Language: Italian\n      Food: Pizza and Pasta\n      Currency: Euro\n      ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What do you know abount Greece?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='\n      I know this:\n      Capital: Athens\n      Language: Greek\n      Food: So

In [20]:
chain = final_prompt | chat

In [29]:
chain.invoke({'country': 'Germany'})


      I know this:
      Capital: Berlin
      Language: German
      Food: Bratwurst and Sauerkraut
      Currency: Euro
      

AIMessage(content='\n      I know this:\n      Capital: Berlin\n      Language: German\n      Food: Bratwurst and Sauerkraut\n      Currency: Euro\n      ', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-3.5-turbo-0125', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--019eedd9-9083-74e2-8f55-cfaea35ac710', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 171, 'output_tokens': 35, 'total_tokens': 206, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [30]:
chain.invoke({'country': 'South Korea'})


      I know this:
      Capital: Seoul
      Language: Korean
      Food: Kimchi and Bibimbap
      Currency: South Korean Won
      

AIMessage(content='\n      I know this:\n      Capital: Seoul\n      Language: Korean\n      Food: Kimchi and Bibimbap\n      Currency: South Korean Won\n      ', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-3.5-turbo-0125', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--019eedda-25ba-7cc1-8f32-f484588c699b', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 172, 'output_tokens': 34, 'total_tokens': 206, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
#  때로는 수천개의 예제를 가지고 있을텐데,  이를 모두 모델에게 줄수 없는 상황이 있을수 있다.
#    이유1) 비용이 많이 든다..  많은 텍스트 땜에.
#    이유2) '허용하는 범위' 라는게 있다 => 모~든 예제들을 모델에게 줄 수는 없다.  
 #          제한이 있다 (context window)

#  그래서 예제를 선별하는 방법에 대해 배워보자

# ExampleSelector

## LengthBasedExapleSelector

In [21]:
from langchain_core.example_selectors.length_based import LengthBasedExampleSelector

In [22]:
# LengthBasedExampleSelector 는 기본적으로
# - 예제(example) 들을 형식화 할 수 있고
# - 예제의 양이 얼마나 되는지를 확인할수 있다.

# 그러면, 사용자가 설정해 놓은 세팅값에 따라 prompt 에 알맞은 예제를 골라준다.

In [23]:
print(examples)

[{'country': 'France', 'answer': '\n      Here is what I know:\n      Capital: Paris\n      Language: French\n      Food: Wine and Cheese\n      Currency: Euro\n      '}, {'country': 'Italy', 'answer': '\n      I know this:\n      Capital: Rome\n      Language: Italian\n      Food: Pizza and Pasta\n      Currency: Euro\n      '}, {'country': 'Greece', 'answer': '\n      I know this:\n      Capital: Athens\n      Language: Greek\n      Food: Souvlaki and Feta Cheese\n      Currency: Euro\n      '}]


In [24]:
example_prompt = PromptTemplate.from_template("Human: {country}\nAI: {answer}")

example_prompt

PromptTemplate(input_variables=['answer', 'country'], input_types={}, partial_variables={}, template='Human: {country}\nAI: {answer}')

In [25]:
# ExampleSelector 준비

example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,  # 포맷팅한 양이 얼마나 되는지 알아야 하기에 example_prompt 필요.
    max_length=10,    # 예제의 양을 얼마나 허용할지 정해주기. max_length= 값 밖의 예제는 cut out 됨.

    # LengthBasedExampleSelector 의 max_length 는  '단어의 개수' 단위    
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    # examples= 대신에 example_selector= 지정.
    example_selector=example_selector,

    suffix="Human: What do you know abount {country}?",
    input_variables=['country']
)

prompt.format(country='Brazil')

'Human: What do you know abount Brazil?'

In [ ]:
# ↑ 선택된 예제가 없다?
# example 들이 하나도 포맷팅 안되어 있다?
# 이유: max_length= 값이 너무 작아서!

In [26]:
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=80  # max_length= 변경
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,

    suffix="Human: What do you know abount {country}?",
    input_variables=['country']
)

print(prompt.format(country='Brazil'))

Human: France
AI: 
      Here is what I know:
      Capital: Paris
      Language: French
      Food: Wine and Cheese
      Currency: Euro
      

Human: What do you know abount Brazil?


In [27]:
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=180  # max_length= 변경
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,

    suffix="Human: What do you know abount {country}?",
    input_variables=['country']
)

print(prompt.format(country='Brazil'))

Human: France
AI: 
      Here is what I know:
      Capital: Paris
      Language: French
      Food: Wine and Cheese
      Currency: Euro
      

Human: Italy
AI: 
      I know this:
      Capital: Rome
      Language: Italian
      Food: Pizza and Pasta
      Currency: Euro
      

Human: Greece
AI: 
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      

Human: What do you know abount Brazil?


## BaseExampleSelector
커스텀 ExampleSelector

In [28]:
from langchain_core.example_selectors.base import BaseExampleSelector

In [29]:
# BaseExampleSelector 의 구현객체를 만드려면
#  상속 받은뒤 select_examples() 과 add_example() 을 반드시 오버라이딩 해주어야 한다.

class RandomExampleSelector(BaseExampleSelector):

    def __init__(self, examples):
        self.examples = examples


    # select_examples()
    # 입력에 따라 어떠한 샘플을 사용할지 select 함.
    
    # 이번 예제에서는 examples 리스트 에서 random 으로 선택하게 하려 함.
    # ※ 이는 얼마든지 복잡하게 만들어 볼수도 있다.
    def select_examples(self, input_variables):
        from random import choice
        return [choice(self.examples)]

    # add_example()
    # Add new example to store.  이미 존재하는 example 에 example 을 추가하는 method
    def add_example(self, example):
        self.examples.append(example)




In [30]:
example_selector = RandomExampleSelector(examples=examples)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,

    suffix="Human: What do you know abount {country}?",
    input_variables=['country']
)

print(prompt.format(country='Brazil'))

Human: France
AI: 
      Here is what I know:
      Capital: Paris
      Language: French
      Food: Wine and Cheese
      Currency: Euro
      

Human: What do you know abount Brazil?


# PromptTemplate 저장/읽어오기
- load_prompt()
- prompt 를 'JSON' 혹은 'YAML' 파일로 만들수 잇다.

## json 파일

In [31]:
with open('prompt.json', 'w') as f:
    f.write("""
  {
    "_type":"prompt",
    "template":"What is the capital of {country}",
    "input_variables":["country"]
  }    
    """)

In [33]:
cat prompt.json


  {
    "_type":"prompt",
    "template":"What is the capital of {country}",
    "input_variables":["country"]
  }    
    

In [34]:
from langchain_core.prompts.loading import load_prompt

In [35]:
# JSON 파일 -> PromptTemplate 객체
prompt = load_prompt("./prompt.json")
prompt

/var/folders/q9/xj42kkbn28g2gdfhyypn9yt80000gn/T/ipykernel_15443/867315639.py:2: LangChainDeprecationWarning: The function `load_prompt` was deprecated in LangChain 1.2.21 and will be removed in 2.0.0. Use `Use `dumpd`/`dumps` from `langchain_core.load` to serialize prompts and `load`/`loads` to deserialize them.` instead.
  prompt = load_prompt("./prompt.json")


PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='What is the capital of {country}')

In [36]:
from langchain_core.load import dumpd, dumps, load, loads

In [37]:
json_str = dumps(prompt)   # Prompt 객체 -> JSON 문자열
json_str

'{"lc": 1, "type": "constructor", "id": ["langchain", "prompts", "prompt", "PromptTemplate"], "kwargs": {"input_variables": ["country"], "template": "What is the capital of {country}", "template_format": "f-string"}, "name": "PromptTemplate"}'

In [38]:
# JSON -> LangChain 객체
load(json_str)

/var/folders/q9/xj42kkbn28g2gdfhyypn9yt80000gn/T/ipykernel_15443/2896249716.py:2: LangChainBetaWarning: The function `load` is in beta. It is actively being worked on, so the API may change.
  load(json_str)
/var/folders/q9/xj42kkbn28g2gdfhyypn9yt80000gn/T/ipykernel_15443/2896249716.py:2: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  load(json_str)


'{"lc": 1, "type": "constructor", "id": ["langchain", "prompts", "prompt", "PromptTemplate"], "kwargs": {"input_variables": ["country"], "template": "What is the capital of {country}", "template_format": "f-string"}, "name": "PromptTemplate"}'

In [39]:
loads(json_str)

/var/folders/q9/xj42kkbn28g2gdfhyypn9yt80000gn/T/ipykernel_15443/4192661131.py:1: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  loads(json_str)
/var/folders/q9/xj42kkbn28g2gdfhyypn9yt80000gn/T/ipykernel_15443/4192661131.py:1: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  loads(json_str)


PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='What is the capital of {country}')

# Caching

In [ ]:
# Caching 을 사용하면 모델의 응답을 저장(cache)할수 있다.
# 예를들어
# 똑같은 질문을 받는 상황이라면 그 때마다 답변생성할 필요 없이
# 이미 캐싱된 답변을 재사용 할수 있는 것이다 -->  비용절감! 

## set_llm_cache(), InMemoryCache

In [40]:
from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache

In [41]:
# 이렇게 세팅하면 LLM 의 모든 response 가 '메모리' 에 저장된다.
set_llm_cache(InMemoryCache())

In [42]:
chat = ChatOpenAI(temperature=0.1)

In [43]:
# 동일한 질문을 두번 해볼거다.  시간측정하는 함수를 준비해보자
import time
from datetime import timedelta

def check_laptime(message):
    start_time = time.time()

    response = chat.invoke(message)

    end_time = time.time()
    elapsed_time = end_time - start_time # 경과시간
    print('▶ 경과시간 %s' % (str(timedelta(seconds = elapsed_time))))
    print(f'{len(response.content)} 글자: {response.content}\n')


In [44]:
check_laptime("How do you make Italian pasta")

▶ 경과시간 0:00:04.021554
1038 글자: To make Italian pasta, you will need the following ingredients:

- 2 cups of all-purpose flour
- 2 large eggs
- Pinch of salt
- Water (if needed)

Here is a step-by-step guide to making Italian pasta:

1. On a clean work surface, pour the flour and create a well in the center.
2. Crack the eggs into the well and add a pinch of salt.
3. Using a fork, gradually mix the eggs into the flour until a dough forms.
4. Knead the dough for about 10 minutes until it becomes smooth and elastic. If the dough is too dry, add a little water. If it is too wet, add a little more flour.
5. Wrap the dough in plastic wrap and let it rest for at least 30 minutes.
6. After resting, roll out the dough using a pasta machine or a rolling pin until it is thin and smooth.
7. Cut the dough into your desired shape, such as fettuccine, spaghetti, or ravioli.
8. Cook the pasta in a large pot of boiling salted water for 2-3 minutes, or until al dente.
9. Drain the pasta and toss it with

In [82]:
# 동일 질문에 대해서는 아까 응답한 내용을 리턴.  LLM 호출 발생.
check_laptime("How do you make Italian pasta")

▶ 경과시간 0:00:00.000998
1037 글자: To make Italian pasta, you will need the following ingredients:

- 2 cups of all-purpose flour
- 2 large eggs
- Pinch of salt
- Water (if needed)

Here is a step-by-step guide to making Italian pasta:

1. On a clean work surface, pour the flour and create a well in the center.
2. Crack the eggs into the well and add a pinch of salt.
3. Using a fork, gradually mix the eggs into the flour until a dough forms.
4. Knead the dough for about 10 minutes until it becomes smooth and elastic. If the dough is too dry, add a little water. If it is too wet, add a little more flour.
5. Wrap the dough in plastic wrap and let it rest for at least 30 minutes.
6. After resting, roll out the dough using a pasta machine or a rolling pin until it is thin and smooth.
7. Cut the dough into your desired shape, such as fettuccine, spaghetti, or ravioli.
8. Cook the pasta in a large pot of boiling salted water for 2-3 minutes or until al dente.
9. Drain the pasta and toss it with 

## set_debug()

In [45]:
from langchain_core.globals import set_debug

In [46]:
# LLM 호출시 발생되는 이벤트에 대한 로그 (?) 같을 것들을 보여준다.
set_debug(True)

In [85]:
chat.invoke("How do you make Italian Pizza")

[llm/start] [llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "Human: How do you make Italian Pizza"
  ]
}
[llm/end] [llm:ChatOpenAI] [5.16s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "To make Italian pizza, you will need the following ingredients:\n\n- 2 1/4 teaspoons active dry yeast\n- 1 1/2 cups warm water\n- 3 1/2 cups all-purpose flour\n- 2 teaspoons salt\n- 2 tablespoons olive oil\n- 1 can of crushed tomatoes\n- 1 clove of garlic, minced\n- 1 teaspoon dried oregano\n- 1 teaspoon dried basil\n- 1/2 teaspoon salt\n- 1/4 teaspoon black pepper\n- 2 cups shredded mozzarella cheese\n- Toppings of your choice (such as pepperoni, mushrooms, bell peppers, etc.)\n\nHere's how to make Italian pizza:\n\n1. In a small bowl, dissolve the yeast in the warm water and let it sit for about 5 minutes until it becomes frothy.\n\n2. In a large mixing bowl, combine the flour and salt. Make a well in the center and pour in the yeast mixture and ol

AIMessage(content="To make Italian pizza, you will need the following ingredients:\n\n- 2 1/4 teaspoons active dry yeast\n- 1 1/2 cups warm water\n- 3 1/2 cups all-purpose flour\n- 2 teaspoons salt\n- 2 tablespoons olive oil\n- 1 can of crushed tomatoes\n- 1 clove of garlic, minced\n- 1 teaspoon dried oregano\n- 1 teaspoon dried basil\n- 1/2 teaspoon salt\n- 1/4 teaspoon black pepper\n- 2 cups shredded mozzarella cheese\n- Toppings of your choice (such as pepperoni, mushrooms, bell peppers, etc.)\n\nHere's how to make Italian pizza:\n\n1. In a small bowl, dissolve the yeast in the warm water and let it sit for about 5 minutes until it becomes frothy.\n\n2. In a large mixing bowl, combine the flour and salt. Make a well in the center and pour in the yeast mixture and olive oil. Stir until a dough forms.\n\n3. Turn the dough out onto a floured surface and knead for about 5-7 minutes, until it becomes smooth and elastic. Place the dough in a greased bowl, cover with a clean towel, and let

In [47]:
set_debug(False)

## SQLiteCache
- 메모리가 아닌 데이터베이스에 캐싱 가능

In [48]:
from langchain_community.cache import SQLiteCache

/var/folders/q9/xj42kkbn28g2gdfhyypn9yt80000gn/T/ipykernel_15443/4239652806.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.cache import SQLiteCache


In [49]:
set_llm_cache(SQLiteCache('cache.db'))

In [50]:
ls -all

total 424
drwxr-xr-x@ 8 leo  staff    256 Jun 22 16:23 ./
drwxr-xr-x@ 4 leo  staff    128 Jun 22 09:26 ../
drwxr-xr-x@ 5 leo  staff    160 Jun 22 16:14 .ipynb_checkpoints/
-rw-r--r--@ 1 leo  staff  86871 Jun 22 12:45 01 Hello LangChain.ipynb
-rw-r--r--@ 1 leo  staff  78372 Jun 22 16:23 02 Model IO.ipynb
-rw-r--r--@ 1 leo  staff   6458 Jun 22 16:22 03 Memory.ipynb
-rw-r--r--@ 1 leo  staff  32768 Jun 22 16:23 cache.db
-rw-r--r--@ 1 leo  staff    124 Jun 22 16:22 prompt.json


In [51]:
chat.invoke("How do you make Italian Pizza")

AIMessage(content='To make an authentic Italian pizza, you will need the following ingredients:\n\n- 2 1/4 cups of all-purpose flour\n- 1 teaspoon of salt\n- 1 teaspoon of sugar\n- 1 packet of active dry yeast\n- 1 cup of warm water\n- 2 tablespoons of olive oil\n- Tomato sauce\n- Fresh mozzarella cheese\n- Fresh basil leaves\n- Optional toppings such as pepperoni, mushrooms, or olives\n\nHere is a step-by-step guide to making Italian pizza:\n\n1. In a large mixing bowl, combine the flour, salt, and sugar. In a separate small bowl, dissolve the yeast in the warm water and let it sit for about 5 minutes until it becomes frothy.\n\n2. Pour the yeast mixture and olive oil into the flour mixture and stir until a dough forms. Knead the dough on a floured surface for about 5-7 minutes until it becomes smooth and elastic.\n\n3. Place the dough in a greased bowl, cover it with a clean kitchen towel, and let it rise in a warm place for about 1-2 hours until it doubles in size.\n\n4. Preheat you

In [52]:
chat.invoke("How do you make Italian Pizza")

/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/Dropbox/K16/PyWork/.venv/lib/python3.12/site-packages/langchain_community/cache.py:265: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(row[0]) for row in rows]


AIMessage(content='To make an authentic Italian pizza, you will need the following ingredients:\n\n- 2 1/4 cups of all-purpose flour\n- 1 teaspoon of salt\n- 1 teaspoon of sugar\n- 1 packet of active dry yeast\n- 1 cup of warm water\n- 2 tablespoons of olive oil\n- Tomato sauce\n- Fresh mozzarella cheese\n- Fresh basil leaves\n- Optional toppings such as pepperoni, mushrooms, or olives\n\nHere is a step-by-step guide to making Italian pizza:\n\n1. In a large mixing bowl, combine the flour, salt, and sugar. In a separate small bowl, dissolve the yeast in the warm water and let it sit for about 5 minutes until it becomes frothy.\n\n2. Pour the yeast mixture and olive oil into the flour mixture and stir until a dough forms. Knead the dough on a floured surface for about 5-7 minutes until it becomes smooth and elastic.\n\n3. Place the dough in a greased bowl, cover it with a clean kitchen towel, and let it rise in a warm place for about 1-2 hours until it doubles in size.\n\n4. Preheat you

## debug, cache 사용 끄기

In [53]:
set_debug(False)
set_llm_cache(None)  # 캐시 사용 안함

# OpenAI 모델 호출 비용 확인

In [54]:
from langchain_community.callbacks.manager import get_openai_callback

In [95]:
with get_openai_callback() as usage:
    # with 블럭 안에서 LLM 호출하면 비용계산이 usage 에 담긴다
    chat.invoke("What is the recipe for soju")
    print('🟦', usage)

🟦 Tokens Used: 222
	Prompt Tokens: 14
		Prompt Tokens Cached: 0
	Completion Tokens: 208
		Reasoning Tokens: 0
Successful Requests: 1
Total Cost (USD): $0.000319


In [96]:
with get_openai_callback() as usage:
    a = chat.invoke("What is the recipe for soju")
    b = chat.invoke("What is the recipe for bibimbap")
    print("\n✅", a.content)
    print("\n✅", b.content)
    print('🟦', usage)


✅ Ingredients:
- 1 cup of rice
- 1 cup of water
- 1 tablespoon of nuruk (fermentation starter)
- 1 tablespoon of yeast

Instructions:
1. Rinse the rice thoroughly and soak it in water for at least 1 hour.
2. Drain the rice and steam it until it is fully cooked.
3. Let the rice cool down to room temperature.
4. In a large bowl, mix the nuruk and yeast with the cooked rice.
5. Cover the bowl with a clean cloth and let it ferment in a warm place for 3-4 days.
6. After the fermentation process is complete, strain the mixture through a cheesecloth to remove any solids.
7. Transfer the liquid to a clean container and let it sit for another 1-2 days to allow the flavors to develop.
8. Your homemade soju is now ready to be enjoyed! Serve chilled and enjoy responsibly.

✅ Bibimbap is a popular Korean dish that consists of a bowl of rice topped with various vegetables, meat, and a fried egg. Here is a basic recipe for bibimbap:

Ingredients:
- 2 cups cooked white rice
- 1 carrot, julienned
- 1 

# 모델 config 저장/불러오기

In [55]:
llm = OpenAI(
    temperature=0.1,
    max_tokens=450,
    model="gpt-4-turbo",
)

In [56]:
llm.save('model.json')

In [57]:
cat model.json

{
    "model_name": "gpt-4-turbo",
    "temperature": 0.1,
    "top_p": 1,
    "frequency_penalty": 0,
    "presence_penalty": 0,
    "n": 1,
    "seed": null,
    "logprobs": null,
    "max_tokens": 450,
    "_type": "openai"
}

In [58]:
from langchain_community.llms.loading import load_llm

In [59]:
chat = load_llm('model.json')

/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/Dropbox/K16/PyWork/.venv/lib/python3.12/site-packages/langchain_community/llms/openai.py:255: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain_community.chat_models import ChatOpenAI`
  warnings.warn(
/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/Dropbox/K16/PyWork/.venv/lib/python3.12/site-packages/langchain_community/llms/openai.py:1089: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain_community.chat_models import ChatOpenAI`
  warnings.warn(
